# Occlusion on Colab (interpretability)

Blanks out one stripe of each test song's spectrogram at a time and records how the model's
confidence in the song's true top-level genre changes (`src/tiny_timbre/occlusion.py`).
No training, only predictions, for the flat (`e0_cnn_flat`) and GCN (`e1_gcn_s0`) models.

**Before you start:**
- Runtime → Change runtime type → **T4 GPU** (CPU also works, just slower: roughly 30–60 min instead of a few).
- The predictions for this check must already be committed in HYPOTHESES.md, and `occlusion.py` must be on `main` (this notebook pulls `main`).

Needs on Drive: `data/mels_medium/` (spectrograms), `checkpoints/e0_cnn_flat.best.pt` and
`checkpoints/e1_gcn_s0.best.pt`. Output goes to Drive's `results/`.

In [ ]:
# 1. Drive (spectrograms, checkpoints, results live here)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/dl-final"   # the team's shared folder (add a shortcut to My Drive if it's missing)

In [ ]:
# 2. Get the latest code
!test -d /content/tiny-timbre || git clone -q https://github.com/mediabymadelyn/tiny-timbre.git /content/tiny-timbre
%cd /content/tiny-timbre
!git checkout -q main && git pull -q
!git log --oneline -1
!test -f src/tiny_timbre/occlusion.py && echo "occlusion.py found" || echo "occlusion.py NOT on main yet: merge it first"

In [ ]:
# 3. Install the locked environment with uv, then check the GPU
!pip install -q uv
!uv sync -q
!uv run python -c "import torch; print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# 4. Data onto Colab's local disk (much faster to read than Drive). Skips anything already copied.
!mkdir -p data
!test -f data/mels_medium/logmel_f16.npy || cp -r "$DRIVE/data/mels_medium" data/    # ~8 GB, a few minutes
!test -f data/fma_metadata/tracks.csv || ( (test -f "$DRIVE/data/fma_metadata.zip" && cp "$DRIVE/data/fma_metadata.zip" /content/) || curl -sSo /content/fma_metadata.zip https://os.unil.cloud.switch.ch/fma/fma_metadata.zip ) && unzip -qn /content/fma_metadata.zip -d data/
!ls -l data/mels_medium    # logmel_f16.npy should be 8268800128 bytes

In [ ]:
# 5. Checkpoints from Drive; results straight to Drive
!mkdir -p "$DRIVE/results"
!ln -sfn "$DRIVE/checkpoints" checkpoints
!ln -sfn "$DRIVE/results" data/results
!ls -l checkpoints/e0_cnn_flat.best.pt checkpoints/e1_gcn_s0.best.pt

## Run

Frequency: 8 bands from low to high pitch. Time: six 5-second pieces of the clip.
30 test songs per top-level genre (fewer if a genre has fewer). Each command prints progress
and writes `results/occlusion_<run>_<axis>.csv` on Drive.

In [ ]:
!uv run python -m tiny_timbre.occlusion --config configs/e0_cnn_flat.yaml configs/e1_gcn_s0.yaml --axis frequency

In [ ]:
!uv run python -m tiny_timbre.occlusion --config configs/e0_cnn_flat.yaml configs/e1_gcn_s0.yaml --axis time

In [ ]:
# 6. Check the four files are there, then download them into data/results/ on your laptop
!ls -l data/results/occlusion_*.csv